<a href="https://colab.research.google.com/github/Hammadullah2/genai/blob/main/notebooks/05_autoregressive/01_lstm/lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🥙 LSTM on Recipe Data

In this notebook, we'll walk through the steps required to train your own LSTM on the recipes dataset

In [23]:
!wget https://raw.githubusercontent.com/Aiven-Labs/demo-opensearch-python/master/full_format_recipes.json

--2026-04-14 05:41:12--  https://raw.githubusercontent.com/Aiven-Labs/demo-opensearch-python/master/full_format_recipes.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 43979776 (42M) [text/plain]
Saving to: ‘full_format_recipes.json.1’

full_format_recipes 100%[===================>]  41.94M  --.-KB/s    in 0.1s    

2026-04-14 05:41:13 (318 MB/s) - ‘full_format_recipes.json.1’ saved [43979776/43979776]



In [24]:
import numpy as np
import json
import re
import string

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, losses

## 0. Parameters <a name="parameters"></a>

In [25]:
VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 100
N_UNITS = 128
VALIDATION_SPLIT = 0.2
SEED = 42
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 3  #changed from 25 to 10 for faster training

## 1. Load the data <a name="load"></a>

In [26]:
import json

# Updated path for Colab
recipe_data = []
try:
    with open("full_format_recipes.json", "r") as json_data_file:
        recipe_data = json.load(json_data_file)
except json.JSONDecodeError as e:
    if "Extra data" in str(e) and e.pos is not None:
        print(f"JSONDecodeError: Extra data found at position {e.pos}. Attempting to parse by removing trailing data.")
        with open("full_format_recipes.json", "r") as json_data_file:
            content = json_data_file.read()
            # Try to find the last valid JSON structure
            # This is a bit heuristic. We assume it's a list or dict.
            # We can try to find the last closing brace/bracket and trim after it.
            last_valid_char_pos = -1
            for i in range(len(content) - 1, -1, -1):
                if content[i] == ']' or content[i] == '}':
                    last_valid_char_pos = i
                    break

            if last_valid_char_pos != -1:
                cleaned_content = content[:last_valid_char_pos + 1]
                try:
                    recipe_data = json.loads(cleaned_content)
                    print("Successfully loaded JSON after trimming extra data.")
                except json.JSONDecodeError as inner_e:
                    print(f"Failed to load JSON even after trimming: {inner_e}")
                    # If still failing, re-raise or handle as a fundamental error
                    raise inner_e
            else:
                print("Could not find a valid closing bracket/brace to trim the JSON. Re-raising original error.")
                raise e
    else:
        # Re-raise other JSON decoding errors
        raise e

# Verify it worked
print(f"Successfully loaded {len(recipe_data)} recipes.")

JSONDecodeError: Extra data found at position 43979775. Attempting to parse by removing trailing data.
Successfully loaded JSON after trimming extra data.
Successfully loaded 20130 recipes.


In [27]:
import os
os.makedirs("./checkpoint", exist_ok=True)
os.makedirs("./models", exist_ok=True)

In [28]:
# Filter the dataset
filtered_data = [
    "Recipe for " + x["title"] + " | " + " ".join(x["directions"])
    for x in recipe_data
    if "title" in x
    and x["title"] is not None
    and "directions" in x
    and x["directions"] is not None
]

In [29]:
# Count the recipes
n_recipes = len(filtered_data)
print(f"{n_recipes} recipes loaded")

20111 recipes loaded


In [30]:
example = filtered_data[9]
print(example)

Recipe for Ham Persillade with Mustard Potato Salad and Mashed Peas  | Chop enough parsley leaves to measure 1 tablespoon; reserve. Chop remaining leaves and stems and simmer with broth and garlic in a small saucepan, covered, 5 minutes. Meanwhile, sprinkle gelatin over water in a medium bowl and let soften 1 minute. Strain broth through a fine-mesh sieve into bowl with gelatin and stir to dissolve. Season with salt and pepper. Set bowl in an ice bath and cool to room temperature, stirring. Toss ham with reserved parsley and divide among jars. Pour gelatin on top and chill until set, at least 1 hour. Whisk together mayonnaise, mustard, vinegar, 1/4 teaspoon salt, and 1/4 teaspoon pepper in a large bowl. Stir in celery, cornichons, and potatoes. Pulse peas with marjoram, oil, 1/2 teaspoon pepper, and 1/4 teaspoon salt in a food processor to a coarse mash. Layer peas, then potato salad, over ham.


## 2. Tokenise the data

In [31]:
# Pad the punctuation, to treat them as separate 'words'
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s


text_data = [pad_punctuation(x) for x in filtered_data]

In [32]:
# Display an example of a recipe
example_data = text_data[9]
example_data

'Recipe for Ham Persillade with Mustard Potato Salad and Mashed Peas | Chop enough parsley leaves to measure 1 tablespoon ; reserve . Chop remaining leaves and stems and simmer with broth and garlic in a small saucepan , covered , 5 minutes . Meanwhile , sprinkle gelatin over water in a medium bowl and let soften 1 minute . Strain broth through a fine - mesh sieve into bowl with gelatin and stir to dissolve . Season with salt and pepper . Set bowl in an ice bath and cool to room temperature , stirring . Toss ham with reserved parsley and divide among jars . Pour gelatin on top and chill until set , at least 1 hour . Whisk together mayonnaise , mustard , vinegar , 1 / 4 teaspoon salt , and 1 / 4 teaspoon pepper in a large bowl . Stir in celery , cornichons , and potatoes . Pulse peas with marjoram , oil , 1 / 2 teaspoon pepper , and 1 / 4 teaspoon salt in a food processor to a coarse mash . Layer peas , then potato salad , over ham . '

In [33]:
# Convert to a Tensorflow Dataset
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)

In [34]:
# Create a vectorisation layer
vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1,
)

In [35]:
# Adapt the layer to the training set
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

In [36]:
# Display some token:word mappings
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")

0: 
1: [UNK]
2: .
3: ,
4: and
5: to
6: in
7: the
8: with
9: a


In [37]:
# Display the same example converted to ints
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())

[  26   16  557    1    8  298  335  189    4 1054  494   27  332  228
  235  262    5  594   11  133   22  311    2  332   45  262    4  671
    4   70    8  171    4   81    6    9   65   80    3  121    3   59
   12    2  299    3   88  650   20   39    6    9   29   21    4   67
  529   11  164    2  320  171  102    9  374   13  643  306   25   21
    8  650    4   42    5  931    2   63    8   24    4   33    2  114
   21    6  178  181 1245    4   60    5  140  112    3   48    2  117
  557    8  285  235    4  200  292  980    2  107  650   28   72    4
  108   10  114    3   57  204   11  172    2   73  110  482    3  298
    3  190    3   11   23   32  142   24    3    4   11   23   32  142
   33    6    9   30   21    2   42    6  353    3 3224    3    4  150
    2  437  494    8 1281    3   37    3   11   23   15  142   33    3
    4   11   23   32  142   24    6    9  291  188    5    9  412  572
    2  230  494    3   46  335  189    3   20  557    2    0    0    0
    0 

## 3. Create the Training Set

In [38]:
# Create the training set of recipes and the same text shifted by one word
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y


train_ds = text_ds.map(prepare_inputs)

## 4. Build the LSTM <a name="build"></a>

In [39]:
inputs = layers.Input(shape=(None,), dtype="int32")
x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x = layers.LSTM(N_UNITS, return_sequences=True)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
lstm = models.Model(inputs, outputs)
lstm.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, None, 100)      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, None, 128)      │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, None, 10000)    │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,407,248 (9.18 MB)

 Trainable params: 2,407,248 (9.18 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
if LOAD_MODEL:
    # model.load_weights('./models/model')
    lstm = models.load_model("./models/lstm", compile=False)

## 5. Train the LSTM <a name="train"></a>

In [41]:
loss_fn = losses.SparseCategoricalCrossentropy()
lstm.compile("adam", loss_fn)

In [42]:
# Create a TextGenerator checkpoint
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }  # <1>

    def sample_from(self, probs, temperature):  # <2>
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]  # <3>
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:  # <4>
            x = np.array([start_tokens])
            y = self.model.predict(x, verbose=0)  # <5>
            sample_token, probs = self.sample_from(y[0][-1], temperature)  # <6>
            info.append({"prompt": start_prompt, "word_probs": probs})
            start_tokens.append(sample_token)  # <7>
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("recipe for", max_tokens=100, temperature=1.0)

In [43]:
# Create a model save checkpoint
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint/checkpoint.weights.h5", # Changed filepath extension
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

# Tokenize starting prompt
text_generator = TextGenerator(vocab)

In [44]:
lstm.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[model_checkpoint_callback, tensorboard_callback, text_generator],
)

Epoch 1/3
628/629 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 5.0430
generated text:
recipe for vegetable cook , apple | garlic , pour medium - large shave of batter third in solids . discard 3 to 15 minutes over be transfer syrup for lemon week on large dry marjoram 3 cup half seconds of vinaigrette in medium - spinach on up ) . drizzle chicken to return hot ( strainer , and boil in each to boil until soft and 

629/629 ━━━━━━━━━━━━━━━━━━━━ 36s 55ms/step - loss: 4.1872
Epoch 2/3
629/629 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 3.1362
generated text:
recipe for cuban individually with cloudy - florets | mince rack 2 minutes of each minutes and chop . cheddar lamb flesh with orange juice , cinnamon glasses . spread dough with the provence , brown pea , turning to taste , reserving to add to 1 1 bacon , stirring for 15 minutes for more 3 minutes . transfer plates . puree . using the electric electric mixer , transfer in a skillet and salt to while drained when kitchen leaves . ( rim longe

In [46]:
# Save the final model
lstm.save("./models/lstm.keras")

## 6. Generate text using the LSTM

In [47]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        print(f"\nPROMPT: {i['prompt']}")
        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, i in zip(p_sorted, i_sorted):
            print(f"{vocab[i]}:   \t{np.round(100*p,2)}%")
        print("--------\n")

In [48]:
info = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /", max_tokens=10, temperature=1.0
)


generated text:
recipe for roasted vegetables | chop 1 / 2 cups



In [49]:
print_probs(info, vocab)


PROMPT: recipe for roasted vegetables | chop 1 /
4:   	38.75%
2:   	37.83000183105469%
3:   	7.789999961853027%
1:   	4.630000114440918%
8:   	2.190000057220459%
--------


PROMPT: recipe for roasted vegetables | chop 1 / 2
cup:   	40.29999923706055%
cups:   	13.579999923706055%
-:   	11.380000114440918%
teaspoon:   	10.319999694824219%
tablespoons:   	2.9700000286102295%
--------



In [50]:
info = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /", max_tokens=10, temperature=0.2
)


generated text:
recipe for roasted vegetables | chop 1 / 4 cup



In [51]:
print_probs(info, vocab)


PROMPT: recipe for roasted vegetables | chop 1 /
4:   	53.0%
2:   	46.9900016784668%
3:   	0.019999999552965164%
1:   	0.0%
8:   	0.0%
--------


PROMPT: recipe for roasted vegetables | chop 1 / 4
cup:   	99.80999755859375%
-:   	0.11999999731779099%
teaspoon:   	0.05999999865889549%
cups:   	0.019999999552965164%
tablespoons:   	0.0%
--------



In [52]:
info = text_generator.generate(
    "recipe for chocolate ice cream |", max_tokens=7, temperature=1.0
)
print_probs(info, vocab)


generated text:
recipe for chocolate ice cream | crush


PROMPT: recipe for chocolate ice cream |
preheat:   	15.989999771118164%
in:   	10.479999542236328%
heat:   	8.09000015258789%
combine:   	7.289999961853027%
bring:   	3.799999952316284%
--------



In [53]:
info = text_generator.generate(
    "recipe for chocolate ice cream |", max_tokens=7, temperature=0.2
)
print_probs(info, vocab)


generated text:
recipe for chocolate ice cream | preheat


PROMPT: recipe for chocolate ice cream |
preheat:   	85.01000213623047%
in:   	10.270000457763672%
heat:   	2.819999933242798%
combine:   	1.6699999570846558%
bring:   	0.05999999865889549%
--------

